# Part 16 (실습) — 관측·평가: 트레이싱과 미니 평가

> **이 노트북의 목표**
> `@traceable`로 추적을 다는 법을 보고, **작은 평가셋 + evaluator**로 에이전트의 정확성·도구 선택을 점수화함. LangSmith는 계정·API 키가 필요하므로, **키 없이도 돌아가는 로컬 미니 평가**로 평가의 원리를 직접 체험함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~15

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai langgraph langsmith

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)
print("준비 완료")

## 1. @traceable — 함수에 추적 달기

`@traceable`은 함수의 입출력·지연을 LangSmith로 보냄. (실제 전송은 아래 환경변수로 켜질 때만 일어남)

In [ ]:
# ─── LangSmith: LLM 애플리케이션 관측·평가 플랫폼 ───
# Client: LangSmith API와 통신하는 클라이언트
# traceable: 함수 실행을 LangSmith에 자동 기록하는 데코레이터
# evaluate: 평가셋으로 체인/에이전트 성능을 자동 측정하는 함수
from langsmith import traceable

@traceable(run_type="tool")
def get_context(question: str) -> str:
    """지식베이스 조회 (데모)"""
    return "반품은 수령 후 7일 이내 가능합니다."

@traceable
def assistant(question: str) -> str:
    """전체 파이프라인 — 하위 traceable이 트리로 매달림"""
    context = get_context(question)
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
    return model.invoke(f"문맥: {context}\n질문: {question}\n문맥 근거로 답해.").content

print(assistant("반품 며칠 안에 해야 해?"))
print("\n→ @traceable이면 LangSmith 연동 시 assistant>get_context 트리로 기록됨(아래 4번).")

## 2. LangChain/LangGraph 자동 추적 — 환경변수 하나

LangChain/LangGraph로 만든 에이전트는 환경변수만 켜면 모든 내부 단계가 자동 기록됨.

In [ ]:
# 실제 추적을 켜려면 (smith.langchain.com에서 키 발급 후):
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "ls-..."
#
# 이 두 줄만 있으면 아래 에이전트의 모든 모델/도구 호출이 LangSmith 대시보드에 트리로 쌓임.
print("환경변수 2개(LANGSMITH_TRACING, LANGSMITH_API_KEY)면 자동 추적 ON")
print("→ 코드 변경 없이 모든 체인·에이전트 실행이 기록됨")

## 3. 평가할 에이전트 준비

간단한 정책 응답 에이전트. 이걸 평가셋으로 점수화할 것.

In [ ]:
@tool
def policy_lookup(topic: str) -> str:
    """정책(반품/환불/배송)을 조회한다. 정책 질문에 사용한다."""
    db = {"반품": "수령 후 7일 이내", "환불": "3영업일 이내", "배송": "당일 출고"}
    for k, v in db.items():
        if k in topic:
            return f"{k}: {v}"
    return "해당 정책 없음"

policy_agent = create_agent(
    model=model, tools=[policy_lookup],
    system_prompt="너는 정책 안내원이야. 정책 질문엔 policy_lookup을 쓰고 간결히 답해.",
)
print("평가 대상 에이전트 준비 완료")

## 4. 미니 평가 — 평가셋 + evaluator (로컬, 키 불필요)

평가셋(입력+기대출력)을 만들고, evaluator(채점기)로 점수화함. LangSmith의 `client.evaluate` 원리를 로컬로 재현.

In [ ]:
# 평가셋: 입력 + 기대 출력 (테스트 케이스)
dataset = [
    {"q": "반품은 며칠 안에?", "expected": "7일", "expected_tool": "policy_lookup"},
    {"q": "환불 얼마나 걸려?", "expected": "3영업일", "expected_tool": "policy_lookup"},
    {"q": "배송 언제 출고?", "expected": "당일", "expected_tool": "policy_lookup"},
]

# evaluator 1: 정확성 — 기대 답이 출력에 포함되는가
def eval_correctness(answer: str, expected: str) -> bool:
    return expected in answer

# evaluator 2: 도구 선택 — 올바른 도구를 불렀는가
def eval_tool_choice(used_tools: list, expected_tool: str) -> bool:
    return expected_tool in used_tools

print("평가셋 3개 + evaluator 2개(정확성, 도구선택) 준비 완료")

In [ ]:
# 평가 실행 — 에이전트를 평가셋 전체에 돌려 점수화
import time

results = []
for ex in dataset:
    t0 = time.time()
    r = policy_agent.invoke({"messages": [("user", ex["q"])]})
    latency = time.time() - t0
    answer = r["messages"][-1].content
    used = [tc["name"] for m in r["messages"] if getattr(m,"tool_calls",None) for tc in m.tool_calls]

    correct = eval_correctness(answer, ex["expected"])
    tool_ok = eval_tool_choice(used, ex["expected_tool"])
    results.append({"q": ex["q"], "correct": correct, "tool_ok": tool_ok, "latency": latency})
    print(f"Q: {ex[\'q\']}")
    print(f"   정확성: {'✅' if correct else '❌'} | 도구선택: {'✅' if tool_ok else '❌'} | 지연: {latency:.2f}s")

In [ ]:
# 종합 점수 — 버전 비교의 기준이 됨
n = len(results)
acc = sum(r["correct"] for r in results) / n
tool_acc = sum(r["tool_ok"] for r in results) / n
avg_lat = sum(r["latency"] for r in results) / n

print("=== 평가 요약 ===")
print(f"정확성:    {acc*100:.0f}%")
print(f"도구 선택: {tool_acc*100:.0f}%")
print(f"평균 지연: {avg_lat:.2f}s")
print("\n→ 프롬프트/모델을 바꾼 뒤 같은 평가셋에 다시 돌리면, 점수로 개선/회귀를 객관 비교할 수 있음.")

> 📌 이것이 LangSmith `client.evaluate(agent, data=..., evaluators=[...])`의 원리임. LangSmith는 이 과정을 클라우드에서 관리하고 버전 간 비교·회귀 플래그를 자동으로 보여줌.

## 5. 회귀 탐지 — 버전 비교

"개선했다"는 느낌이 아니라 점수로 확인. 나쁜 프롬프트로 바꾸면 점수가 떨어지는 걸 봄.

In [ ]:
# 일부러 나쁜 버전 — 도구 쓰지 말라고 지시 (추측하게)
bad_agent = create_agent(
    model=model, tools=[policy_lookup],
    system_prompt="너는 안내원이야. 도구 쓰지 말고 네 지식으로 추측해서 답해.",
)

bad_correct = 0
for ex in dataset:
    r = bad_agent.invoke({"messages": [("user", ex["q"])]})
    if eval_correctness(r["messages"][-1].content, ex["expected"]):
        bad_correct += 1

print(f"좋은 버전 정확성: {acc*100:.0f}%")
print(f"나쁜 버전 정확성: {bad_correct/len(dataset)*100:.0f}%")
print("→ 평가셋이 있으니 '어느 버전이 나은지'를 점수로 판단. 출시 전 회귀를 잡음.")

## ✏️ 미니 실습

**과제**: 평가셋에 "교환 정책" 케이스를 추가하고(`policy_lookup`엔 '교환'이 없음 → 실패 사례), 평가를 다시 돌려 정확성이 떨어지는 걸 확인하기. 이것이 "운영에서 발견한 실패를 평가셋에 추가"하는 과정의 축소판.

In [ ]:
# TODO: dataset에 {"q": "교환 며칠 안에?", "expected": "14일", "expected_tool": "policy_lookup"} 추가 후 재평가
# → policy_lookup DB에 '교환'이 없어 실패 → 이걸 보고 도구를 보강하는 게 개선 루프

<details><summary>👉 정답</summary>

```python
dataset.append({"q": "교환 며칠 안에?", "expected": "14일", "expected_tool": "policy_lookup"})
# 재평가 → '교환' 케이스 실패 발견 → policy_lookup의 db에 "교환":"14일 이내" 추가하면 개선
# 이것이 운영 관측 → 실패 수집 → 평가셋 보강 → 개선의 순환 (16-5)
```
</details>

## 정리

| 절 | 내용 | 핵심 |
|---|---|---|
| 1 | `@traceable` | 함수에 추적 |
| 2 | 자동 추적 | 환경변수 2개 |
| 3~4 | 평가셋+evaluator | 정확성·도구선택 점수화 |
| 5 | 회귀 탐지 | 버전 비교 |

### 3줄 요약
1. `@traceable`/환경변수로 모든 단계를 트레이스로 기록(관측).
2. 평가셋+evaluator로 정확성·도구선택을 점수화(평가).
3. 같은 평가셋 반복으로 버전 간 비교·회귀 탐지 — 느낌이 아니라 점수로.

### ▶ 다음 (Part 17)
만든 시스템을 실제 세상으로 — **배포와 신뢰성**: 영속 상태, 오류 처리·재시도, 안전장치, 비용 관리.